# Этап 2 — baseline fine-tuning VisDrone-4NN
Выберите в Colab **Runtime → Change runtime type → T4 GPU**. Один процесс обучает одну модель.
Используются готовые `data-v1/train.json` и `validation.json`; подготовка и holdout здесь не запускаются.
Сначала выполните smoke test и проверьте результаты. Полное обучение запускается отдельной ячейкой вручную.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Клонируется ваш репозиторий с сохранёнными изменениями этапа 2. Существующий checkout автоматически не обновляется.
Для точного повторения используйте сохранённый commit и `environment.txt`; при незакоммиченных изменениях дополнительно сохраняется `source.zip`.


In [ ]:
from pathlib import Path
import subprocess
import sys
REPO = Path('/content/visdrone-4nn')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/quovadi/visdrone-4nn.git', str(REPO)], check=True)
%cd /content/visdrone-4nn
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
assert Path('training/run.py').exists(), 'Нужен checkout с реализацией этапа 2'
%pip install -r requirements.txt


При сообщении pip о необходимости перезапуска runtime перезапустите его и повторите подключение Drive и переход в REPO.
Синтетические тесты не загружают веса. Реестр проверяет наличие тегов в установленном timm; реальная загрузка весов происходит в smoke test.


In [ ]:
import torch
print('torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Включите T4 GPU для Colab smoke test'
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'training.models'], check=True)


In [ ]:
import json
MODEL = 'convnext_nano'  # convnext_base | convnext_nano | vit_base | tinyvit
ROOT = Path('/content/drive/MyDrive/VisDrone')
INDICES = Path('/content/drive/MyDrive/visdrone-results/data-v1')
OUTPUT = Path('/content/drive/MyDrive/visdrone-results/baseline-v1')
BATCH_SIZE = 8
WORKERS = 2
EPOCHS = 10
LR = 1e-4
SEED = 42
TRAIN_SIZE = None  # None = весь train; для предварительного запуска, например, 4096
assert ROOT.is_dir()
for name in ('train', 'validation'):
    assert (INDICES / f'{name}.json').is_file()
# data-v1 не создаётся и не изменяется. Holdout не открывается.


**Необязательно: локальная копия изображений.** Чтение множества crop из Drive может быть медленным.
По умолчанию следующая ячейка ничего не копирует. Установите `COPY_IMAGES_TO_LOCAL=True` только если согласны скопировать изображения train и val в `/content/` и проверили свободное место (`!df -h /content`).
Индексы и результаты остаются на Drive. При обрыве локальная копия может исчезнуть; её можно создать заново или снова использовать ROOT на Drive.


In [ ]:
COPY_IMAGES_TO_LOCAL = False  # Явное согласие пользователя на копирование
if COPY_IMAGES_TO_LOCAL:
    import shutil
    LOCAL_ROOT = Path('/content/VisDrone')
    print('Свободно байт:', shutil.disk_usage('/content').free)
    for split in ('VisDrone2019-DET-train', 'VisDrone2019-DET-val'):
        target = LOCAL_ROOT / split / 'images'
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(ROOT / split / 'images', target)  # отказ при уже существующей папке
    ROOT = LOCAL_ROOT
print('Чтение изображений из', ROOT)


## Обязательный короткий smoke test одной модели
Одна эпоха, максимум два batch train и два batch validation. Метрики этого запуска не являются оценкой качества.
Новый запуск получает уникальную директорию с префиксом `smoke-`. Веса скачиваются в обычный cache timm/Hugging Face; ошибки загрузки не подменяются случайной инициализацией.


In [ ]:
def base_command():
    return [sys.executable, '-m', 'training.run', '--model', MODEL,
            '--root', str(ROOT), '--indices', str(INDICES), '--output', str(OUTPUT),
            '--batch-size', str(BATCH_SIZE), '--workers', str(WORKERS),
            '--epochs', str(EPOCHS), '--lr', str(LR), '--seed', str(SEED), '--device', 'cuda']

before = set((OUTPUT / MODEL).glob('smoke-*'))
subprocess.run(base_command() + ['--smoke'], check=True)
created = set((OUTPUT / MODEL).glob('smoke-*')) - before
assert len(created) == 1
SMOKE_RUN = created.pop()
print('SMOKE_RUN =', SMOKE_RUN)


In [ ]:
history = json.loads((SMOKE_RUN / 'metrics.json').read_text())
print(json.dumps(history, indent=2))
for name in ('best.pt', 'last.pt', 'config.json', 'data.json', 'provenance.json', 'source.zip'):
    assert (SMOKE_RUN / name).is_file(), name
assert history[0]['train']['samples'] > 0 and history[0]['validation']['samples'] > 0
# Проверка безопасного повторного resume завершённого smoke: эпохи не добавляются.
subprocess.run([sys.executable, '-m', 'training.run', '--resume', str(SMOKE_RUN),
                '--root', str(ROOT)], check=True)


Перед полным запуском просмотрите реальные crop и сопоставление классов, число примеров, loss и сохранённые состояния.
Для маленькой smoke-выборки отсутствие некоторых классов нормально: macro-F1 всё равно усредняется по всем 10 классам.
Pretrained resize + center crop может обрезать края вытянутых объектов; оцените это визуально.


In [ ]:
import matplotlib.pyplot as plt
from training.config import TrainConfig
from training.data import make_datasets
from training.models import make_transforms
from data.visdrone import CLASSES
cfg = TrainConfig(**json.loads((SMOKE_RUN / 'config.json').read_text()))
from dataclasses import replace
cfg = replace(cfg, root=str(ROOT))
data_config = json.loads((SMOKE_RUN / 'data.json').read_text())['resolved_data_config']
train_ds, _, _ = make_datasets(cfg, *make_transforms(data_config))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
mean = torch.tensor(data_config['mean'])[:, None, None]
std = torch.tensor(data_config['std'])[:, None, None]
for i in range(4):
    record = train_ds.records[i % len(train_ds)]
    crop = train_ds.raw_crop(i % len(train_ds))
    processed = train_ds.transform(crop)
    axes[0, i].imshow(crop)
    axes[0, i].set_title(f'{CLASSES[record.label]}, box={record.box}')
    axes[1, i].imshow((processed * std + mean).clamp(0, 1).permute(1, 2, 0))
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()
del train_ds


## Полное обучение — отдельное ручное действие
После успешного smoke и визуальной проверки установите `START_FULL_TRAINING=True`.
Запуск создаёт новый `full-*`; веса smoke не используются. Для четырёх моделей повторяйте последовательно, меняя MODEL и проверяя smoke для каждой.
При OOM выберите меньший batch size и начните новый эксперимент; для итогового сравнения используйте одинаковый batch size всех четырёх моделей.


In [ ]:
START_FULL_TRAINING = False
if START_FULL_TRAINING:
    assert (SMOKE_RUN / 'summary.json').is_file()
    smoke_config = json.loads((SMOKE_RUN / 'config.json').read_text())
    assert smoke_config['model'] == MODEL, 'Сначала smoke выбранной модели'
    command = base_command()
    if TRAIN_SIZE is not None:
        command += ['--train-size', str(TRAIN_SIZE)]
    subprocess.run(command, check=True)
else:
    print('Полное обучение не запущено. Проверьте smoke и включите флаг вручную.')


## Возобновление после отключения Colab
Подключите Drive, восстановите checkout/версии пакетов, ROOT и выполните только ячейку ниже.
Укажите сохранённый RUN_DIR из вывода или выберите нужный `full-*` на Drive. Не запускайте новое полное обучение вместо resume.
Восстанавливаются модель, AdamW, scheduler, AMP scaler и состояния RNG. Продолжение идёт с границы последней полностью сохранённой эпохи; незавершённая эпоха повторяется.
План эпох и гиперпараметры менять при resume нельзя. Исходные код/версии и SHA-256 индексов проверяются.
При переносе изображений разрешено менять только ROOT (данные должны быть идентичны). Checkpoint загружайте только из собственных доверенных запусков.


In [ ]:
RESUME_DIR = ''  # Например /content/drive/MyDrive/visdrone-results/baseline-v1/convnext_nano/full-...
if RESUME_DIR:
    subprocess.run([sys.executable, '-m', 'training.run', '--resume', RESUME_DIR,
                    '--root', str(ROOT)], check=True)


## Кривые завершённого или прерванного запуска
Выбирайте каталог нужного эксперимента, не смешивайте smoke и full.


In [ ]:
PLOT_RUN = SMOKE_RUN  # Замените на Path полного эксперимента
rows = json.loads((Path(PLOT_RUN) / 'metrics.json').read_text())
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ('loss', 'accuracy', 'macro_f1')):
    for split in ('train', 'validation'):
        ax.plot([r['epoch'] for r in rows], [r[split][metric] for r in rows], label=split)
    ax.set(xlabel='epoch', ylabel=metric)
    ax.legend()
plt.tight_layout()
plt.show()


### Ограничения этапа 2
Все архитектуры получают вход 224×224, но normalization/crop_pct соответствуют их весам. TinyViT-5M использует distillation и предобучение ImageNet-22k→1k; остальные выбранные веса — ImageNet-1k с разными рецептами. Это фактор помимо архитектуры и размера.
Дисбаланс не компенсируется: общий baseline использует обычную CrossEntropyLoss, полный список объектов и равновероятное перемешивание. Мелкие объекты не исключаются; bbox остаются в исходных индексах.
В Colab нужно проверить загрузку весов всех четырёх моделей, CUDA AMP/scaler, память T4, скорость I/O, реальный resume и сохранность Drive.
Битовая идентичность между разными GPU/драйверами не гарантируется; детерминированные неподдерживаемые операции завершаются ошибкой.
Здесь нет обучения атак/защит, финальной оценки holdout или профилирования этапа 3.
